# Decision Surface Geometry

**GeoLatent** renders the true 3-D decision boundary of any classifier by projecting
the feature space to three dimensions, building a prediction mesh, and drawing
probability isosurfaces via inverse-transform — not axis-aligned slices.

This notebook shows the technique on three real-world datasets across different
model families.

In [ ]:
!pip install -q geolatent
# After first run: Runtime > Restart session, then re-run from the next cell

In [ ]:
import numpy as np
import plotly.io as pio
from sklearn.datasets import load_wine, load_breast_cancer, load_digits
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC

from geolatent import visualize_decision_geometry, DARK_SCIENTIFIC

pio.renderers.default = "colab"

---
## Wine — RBF SVM

13 chemical measurements, 3 cultivar classes.  The sensitivity projection finds
the directions in feature space where the SVM's decision function changes fastest,
producing axes that are model-driven rather than data-driven.

In [ ]:
wine = load_wine()
svm = SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=0).fit(
    wine.data, wine.target
)

visualize_decision_geometry(
    svm, wine.data, wine.target,
    projection_method="sensitivity",
    feature_names=list(wine.feature_names),
    class_names=dict(enumerate(wine.target_names)),
    show_confidence=True,
    show_centroids=True,
    show_ellipsoids=True,
    title="Wine — RBF SVM (sensitivity projection)",
).show()

PCA on the same model for comparison — the axes now reflect data variance
rather than decision sensitivity.

In [ ]:
visualize_decision_geometry(
    svm, wine.data, wine.target,
    projection_method="pca",
    feature_names=list(wine.feature_names),
    class_names=dict(enumerate(wine.target_names)),
    show_confidence=True,
    show_centroids=True,
    show_ellipsoids=True,
    title="Wine — RBF SVM (PCA projection)",
).show()

---
## Breast Cancer — Gradient Boosting

30 tumour morphology features, binary classification.  Gradient Boosting builds
non-convex, step-function boundaries; the sensitivity projection exposes which
features dominate the ensemble's splits.

In [ ]:
cancer = load_breast_cancer()
gbm = GradientBoostingClassifier(n_estimators=150, max_depth=3, random_state=0).fit(
    cancer.data, cancer.target
)

visualize_decision_geometry(
    gbm, cancer.data, cancer.target,
    projection_method="sensitivity",
    feature_names=list(cancer.feature_names),
    class_names={0: "Malignant", 1: "Benign"},
    show_confidence=True,
    show_centroids=True,
    show_ellipsoids=True,
    title="Breast Cancer — Gradient Boosting (sensitivity projection)",
).show()

---
## Handwritten Digits — Random Forest

64 pixel intensities (8×8 images), digits 0–4.  The sensitivity axes surface
the pixel regions each tree ensemble splits on — effectively a visual feature
importance map embedded in the geometry.

In [ ]:
digits = load_digits()
mask = digits.target < 5
X_dig, y_dig = digits.data[mask], digits.target[mask]

rf = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_dig, y_dig)
pixel_names = [f"px_{i // 8}_{i % 8}" for i in range(64)]

visualize_decision_geometry(
    rf, X_dig, y_dig,
    projection_method="sensitivity",
    feature_names=pixel_names,
    class_names={i: f"Digit {i}" for i in range(5)},
    show_confidence=False,
    show_centroids=True,
    show_ellipsoids=True,
    title="Digits 0–4 — Random Forest (sensitivity on 64 pixels)",
).show()

---
## Iris — Three Models Side by Side

The classic 4-feature, 3-class dataset.  Shown with three different model families
to illustrate how the decision geometry changes with inductive bias.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

iris = load_iris()
names = dict(enumerate(iris.target_names))
features = list(iris.feature_names)

models = [
    ("Logistic Regression", LogisticRegression(max_iter=500, random_state=0)),
    ("RBF SVM",             SVC(kernel="rbf", probability=True, random_state=0)),
    ("MLP (64, 32)",        MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=0)),
]

for label, clf in models:
    clf.fit(iris.data, iris.target)
    visualize_decision_geometry(
        clf, iris.data, iris.target,
        projection_method="sensitivity",
        feature_names=features,
        class_names=names,
        show_confidence=True,
        show_centroids=True,
        show_ellipsoids=True,
        title=f"Iris — {label}",
    ).show()